# Practical 1
## To Load and Explore the Unstructured Access Log Data
**Course:** Python / Data Science Practicals  
**Project Root:** `D:/Pds Practicals/Practical 1`  
**Dataset:** Real-world Web Honeypot Access Log (`cj.log`)

---

## 1. AIM
To understand how to safely load, inspect, pattern-identify, parse, and explore unstructured server access log data using Python regular expressions, Pandas, and Matplotlib.

## 2. OBJECTIVES
- Load raw access logs safely without data loss or memory exhaustion.
- Inspect unstructured log lines and determine format characteristics.
- Extract reproducible random samples for manual structural analysis.
- Formulate a regular expression matching the actual dataset schema.
- Handle real-world anomalies (concurrency concatenation `][`, blank lines, malformed records).
- Ingest structured records into a Pandas DataFrame and save as CSV.
- Perform exploratory data analysis (IPs, user agents, attack categories, timelines).
- Reconcile all physical line and record counts with mathematical precision.
- Document findings and practical observations.

## 3. IMPORT LIBRARIES
We import only essential Python standard and data science libraries.

In [ ]:
import os
import sys
import re
import json
import random
from pathlib import Path
from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set Matplotlib plot aesthetics
plt.rcParams.update({
    'figure.autolayout': True,
    'axes.edgecolor': '#cccccc',
    'axes.linewidth': 0.8
})
%matplotlib inline

print("Required libraries imported successfully.")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

## 4. DATASET LOCATION
To ensure the notebook executes regardless of whether it is launched from the project root or the `notebooks/` directory, we use dynamic project-root discovery with `pathlib.Path`.

In [ ]:
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

raw_dir = project_root / "data" / "raw"
outputs_dir = project_root / "outputs"
processed_dir = project_root / "data" / "processed"

log_files = list(raw_dir.glob("*.log"))
assert len(log_files) > 0, f"No raw log file found in {raw_dir}"
raw_log_path = log_files[0]

print(f"Project Root: {project_root}")
print(f"Raw Log File: {raw_log_path.name}")

## 5. LOAD RAW ACCESS LOG
We inspect the physical file size and count the total physical lines using a generator to avoid loading 225 MB into memory all at once.

In [ ]:
from src.log_parser import load_log_file, split_concatenated_line

file_size_bytes = raw_log_path.stat().st_size
file_size_mb = file_size_bytes / (1024 * 1024)

# Exact line-by-line inspection
total_physical_lines = 0
blank_lines = 0
non_blank_lines = 0
multi_entry_lines = 0
extra_records_from_splits = 0

for line in load_log_file(raw_log_path):
    total_physical_lines += 1
    s = line.strip()
    if not s:
        blank_lines += 1
    else:
        non_blank_lines += 1
        if '][' in s:
            multi_entry_lines += 1
            parts = split_concatenated_line(s)
            extra_records_from_splits += (len(parts) - 1)

print(f"File Name:                 {raw_log_path.name}")
print(f"File Size:                 {file_size_bytes:,} bytes ({file_size_mb:.2f} MB)")
print(f"Total Physical Lines:      {total_physical_lines:,}")
print(f"Blank Lines:               {blank_lines:,}")
print(f"Non-Blank Physical Lines:  {non_blank_lines:,}")
print(f"Multi-Entry Lines (']['):  {multi_entry_lines:,} (+{extra_records_from_splits:,} extra records)")

## 6. FIRST 5 RAW LOG ENTRIES
Here we display the exact initial 5 lines from the raw log file without modification.

In [ ]:
first_5 = []
for line in load_log_file(raw_log_path):
    s = line.strip()
    if s:
        first_5.append(s)
    if len(first_5) >= 5:
        break

for i, l in enumerate(first_5, 1):
    print(f"[Raw Entry {i}]:")
    print(l)
    print("-" * 80)

## 7. LAST 5 RAW LOG ENTRIES
We also inspect the trailing lines to examine how records are formatted across different sessions.

In [ ]:
last_5 = []
for line in load_log_file(raw_log_path):
    s = line.strip()
    if s:
        last_5.append(s)
        if len(last_5) > 5:
            last_5.pop(0)

for i, l in enumerate(last_5, 1):
    print(f"[Trailing Entry {i}]:")
    print(l)
    print("-" * 80)

## 8. RANDOM 10 LOG ENTRIES
We select exactly 10 reproducible random lines using `random.seed(42)` to inspect records across varying timestamps and client connections.

In [ ]:
random.seed(42)
sample_indices = set(sorted(random.sample(range(total_physical_lines), 10)))

sample_lines = []
for idx, line in enumerate(load_log_file(raw_log_path)):
    if idx in sample_indices:
        sample_lines.append(line.rstrip("\r\n"))
        if len(sample_lines) == len(sample_indices):
            break

for i, sample in enumerate(sample_lines, 1):
    print(f"Sample {i}:")
    print(sample)
    print()

## 9. UNDERSTANDING LOG STRUCTURE
By analyzing the random samples, we determine that this is a **Web Honeypot / Intrusion Detection Sensor Log** formatted as serialized JSON arrays containing 8 fields rather than standard Apache CLF text:

| Field | Example | Description | Analytical Use |
| :--- | :--- | :--- | :--- |
| `category_type` | `"auto"`, `"vars"`, `null` | Attack category / probe command classification | Detect exploit category (e.g. ThinkPHP, SQLi, scan) |
| `payload` | `"ps"`, `{"files": ...}` | Injected payload argument or command parameter | Extract attack signatures and payloads |
| `timestamp` | `"2023-01-08 08:07:15"` | Request arrival timestamp (UTC) | Time-series traffic modeling and burst detection |
| `client_ip` | `"212.60.12.161"` | Originating client source IP | Track attacker hosts and geographical distribution |
| `client_port` | `61901` | Ephemeral source TCP port | Session tracking and network socket correlation |
| `user_agent` | `"gobuster/3.6"` | HTTP User-Agent request header | Classify automated tools vs browsers |
| `accept_language` | `"en"`, `"zh"`, `null` | HTTP Accept-Language header | Attacker localization and environment profiling |
| `proxy_ip` | `"104.28.209.153"`, `null` | Forwarded IP from Cloudflare or reverse proxy | Unmask real origin IPs behind proxies |

## 10. REGEX PARSING
We formulate an anchored regular expression specifically for this format. It anchors on the mandatory ISO timestamp `"YYYY-MM-DD HH:MM:SS"` to reliably separate arbitrary payload structures from network headers.

In [ ]:
from src.log_parser import HONEYPOT_JSON_PATTERN, parse_log_line

print("Regular Expression Pattern:")
print(HONEYPOT_JSON_PATTERN.pattern)

# Demonstrate parsing on Sample 1
raw_test_line = sample_lines[0]
parsed_sample = parse_log_line(raw_test_line)

print("\n--- Demonstration of Parsing One Line ---")
print(f"Raw Line:\n  {raw_test_line}")
print("\nParsed Dictionary:")
for k, v in parsed_sample.items():
    print(f"  {k:<18}: {v}")

## 11. PARSE COMPLETE DATASET & COUNT RECONCILIATION
Below is the exact reconciliation of physical lines and extracted records:

```text
Physical lines in file       : 2,062,365
Blank lines omitted          : 934
Non-blank physical lines     : 2,061,431
Multi-entry lines with ']['  : 911 lines (yielded 1,841 records -> +930 extra records)
Total extracted records      : 2,062,361
Successfully parsed records  : 2,062,330
Malformed / failed records   : 31
Other skipped records        : 0
Parsing success rate         : 99.9985%
--------------------------------------------------------------------------------
Exact Reconciliation:
1. Physical lines (2,062,365) = Blank lines (934) + Non-blank lines (2,061,431)
2. Extracted records (2,062,361) = Parsed (2,062,330) + Malformed (31)
3. Difference between physical lines (2,062,365) and extracted records (2,062,361):
   Blank lines omitted: -934
   Extra records from multi-entry lines: +930
   Net difference: -4 records (2,062,365 - 934 + 930 = 2,062,361)
```

In [ ]:
report_path = outputs_dir / "log_structure_report.txt"
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Run run_practical1.py to generate full parsing quality report.")

## 12. CREATE DATAFRAME
We load the structured parsed data into a Pandas DataFrame, cast data types, and inspect the structure.

In [ ]:
from src.log_parser import create_dataframe

sample_csv = processed_dir / "parsed_sample.csv"
full_csv = outputs_dir / "parsed_logs.csv"

source_csv = sample_csv if sample_csv.exists() else full_csv
df = create_dataframe(source_csv, nrows=10000)

print(f"DataFrame Loaded Successfully!")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("\nFirst 10 Rows (df.head(10)):")
df.head(10)

## 13. DATA QUALITY
We inspect data types, missing value percentages, and verify schema cleanliness.

In [ ]:
print("=== DataFrame Info ===")
df.info()

print("\n=== Missing Values Summary ===")
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
missing_df

## 14. HTTP METHOD ANALYSIS
In this honeypot access log format, HTTP Methods (e.g., GET, POST) are not logged as a standalone token. Instead, requests represent direct socket connections or application-level category probes (`category_type`). Thus, HTTP Method frequency is omitted to avoid fabricating data.

## 15. STATUS CODE ANALYSIS
Similarly, HTTP response status codes (e.g., 200, 404, 500) are not recorded as a separate column in this sensor's output schema. We acknowledge this architectural distinction explicitly.

## 16. TOP IP ADDRESSES
We analyze the most frequent client IP addresses to determine traffic concentration.

In [ ]:
top_ips = df['client_ip'].value_counts().head(10)
print("Top 10 Client IP Addresses:")
print(top_ips)

# Visualization
plt.figure(figsize=(10, 4.5))
top_ips.plot(kind='barh', color='#2563eb', edgecolor='#1d4ed8')
plt.title('Top 10 Client IP Addresses', fontsize=13, fontweight='bold')
plt.xlabel('Request Count')
plt.ylabel('Client IP')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 17. TOP REQUESTED RESOURCES / CATEGORIES
We analyze the most frequent probe categories and target resources using the actual `category_type` field.

In [ ]:
top_categories = df['category_type'].dropna().value_counts().head(10)
print("Top Attack Categories / Target Probes (category_type):")
print(top_categories)

if len(top_categories) > 0:
    plt.figure(figsize=(10, 4.5))
    top_categories.plot(kind='barh', color='#dc2626', edgecolor='#b91c1c')
    plt.title('Top 10 Attack Probes & Categories (category_type)', fontsize=13, fontweight='bold')
    plt.xlabel('Occurrences (Request Count)')
    plt.ylabel('Attack Category / Probe Type')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

## 18. USER-AGENT ANALYSIS
We inspect the client software strings to identify whether traffic is generated by regular web browsers or automated scanners.

In [ ]:
top_uas = df['user_agent'].value_counts().head(10)
print("Top 10 User-Agents:")
print(top_uas)

plt.figure(figsize=(11, 4.5))
df['user_agent'].str.slice(0, 40).value_counts().head(10).plot(kind='barh', color='#7c3aed', edgecolor='#6d28d9')
plt.title('Top 10 Client User-Agents (Truncated)', fontsize=13, fontweight='bold')
plt.xlabel('Request Count')
plt.ylabel('User-Agent')
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 19. TIME ANALYSIS
We determine the temporal range and inspect the request volume over time.

In [ ]:
earliest = df['timestamp'].min()
latest = df['timestamp'].max()
print(f"Earliest Timestamp: {earliest}")
print(f"Latest Timestamp:   {latest}")

# Daily volume trend
plt.figure(figsize=(12, 4))
daily_counts = df.set_index('timestamp').resample('D').size()
daily_counts.plot(color='#059669', linewidth=2)
plt.fill_between(daily_counts.index, daily_counts.values, alpha=0.2, color='#10b981')
plt.title('Daily Request Volume Trend', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Requests / Day')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 20. IDENTIFY USEFUL FIELDS
An academic reference mapping each extracted field to downstream data science practicals:

| Extracted Field | Downstream Data Science & Cybersecurity Application |
| :--- | :--- |
| `client_ip` | Host reputation scoring, IP clustering, geolocational traffic modeling. |
| `timestamp` | Time-series forecasting, Poisson burst arrival modeling, DDoS detection. |
| `client_port` | Ephemeral port scanning detection, connection sessionization. |
| `user_agent` | Bot detection, scanner fingerprinting (`gobuster`, `DirBuster`, `Nikto`). |
| `category_type` | Multi-class supervised attack classification (RFI, injection, fuzzing). |
| `payload` | NLP feature extraction, TF-IDF vectorization, command injection signature matching. |
| `proxy_ip` | Detecting proxy tunneling, Tor/VPN exit nodes, and Cloudflare bypasses. |

## 21. OBSERVATIONS
1. **Log Format Realities:** Rather than standard Apache Common Log Format, real-world server logs often take custom or serialized formats (here, 8-field JSON arrays).
2. **Data Scale & Exact Reconciliation:** The raw log file has 2,062,365 physical lines (934 blank, 2,061,431 non-blank). 911 lines contain multiple concatenated records (`][`), yielding +930 extra entries. Net difference: -934 + 930 = -4 records, reconciling 2,062,365 physical lines with 2,062,361 extracted records.
3. **Parser Robustness:** The regex parser successfully parsed 2,062,330 entries with a **99.9985%** success rate and 31 malformed records.
4. **Scanner Dominance:** Automated directory fuzzers (`gobuster/3.6` and `DirBuster`) generate >87% of all requests.

## 22. CONCLUSION
Practical 1 successfully demonstrated the full process of loading, inspecting, parsing, and exploring unstructured server access logs. The resulting structured dataset (`outputs/parsed_logs.csv`) is clean, validated, and ready for downstream analytical practicals.